In [1]:
print("Clinical Data Cleaning Project - Setup Complete")

Clinical Data Cleaning Project - Setup Complete


# Clinical Data Cleaning Project

## Step 3: Synthetic Dataset Generation

**Purpose:** Generate a synthetic clinical study dataset containing 100 subjects and deliberately seeded data-quality issues.

**Data status:** All records are artificial and created for educational purposes. No real patient data is used.

**Variables:** SUBJID, SITE, ICDT, VISDT, AGE, SEX, HT, WT, SBP, DBP, HGB, AETERM.

**Next step:** Apply documented edit checks and generate a discrepancy log.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Reproducible results
rng = np.random.default_rng(20261021)

# Create project folders in Colab
PROJECT_DIR = Path("/content/Clinical-Data-Cleaning-Project")
DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Generate 100 synthetic subjects
n = 100

consent_dates = (
    pd.to_datetime("2025-01-01")
    + pd.to_timedelta(
        rng.integers(0, 365, n), unit="D"
    )
)

df = pd.DataFrame({
    "SUBJID": [f"S{i:03d}" for i in range(1, n + 1)],
    "SITE": rng.choice([101, 102, 103], n),
    "ICDT": consent_dates,
    "VISDT": consent_dates + pd.to_timedelta(
        rng.integers(0, 15, n), unit="D"
    ),
    "AGE": rng.integers(18, 76, n),
    "SEX": rng.choice(["M", "F"], n),
    "HT": np.round(rng.uniform(145, 195, n), 1),
    "WT": np.round(rng.uniform(45, 120, n), 1),
    "SBP": rng.integers(95, 161, n),
    "DBP": rng.integers(60, 101, n),
    "HGB": np.round(rng.uniform(10, 16, n), 1),
    "AETERM": rng.choice(
        ["", "Headache", "Nausea", "Fatigue", "Dizziness"],
        n,
        p=[0.55, 0.12, 0.12, 0.13, 0.08]
    )
})

# Ensure baseline DBP is below SBP
df["DBP"] = np.minimum(df["DBP"], df["SBP"] - 10)

# Preserve the generated baseline for testing
baseline_df = df.copy(deep=True)

# Introduce deliberate data-quality issues
seeded_issues = []

def seed_error(row, column, value, description):
    old_value = df.loc[row, column]
    original_id = baseline_df.loc[row, "SUBJID"]

    df.loc[row, column] = value

    seeded_issues.append({
        "ROW_INDEX": row,
        "SUBJID_AT_SEEDING": original_id,
        "VARIABLE": column,
        "ORIGINAL_VALUE": str(old_value),
        "SEEDED_VALUE": str(value),
        "ISSUE_DESCRIPTION": description
    })

# Missing values
seed_error(2, "AGE", np.nan, "Missing age")
seed_error(11, "HT", np.nan, "Missing height")
seed_error(24, "HGB", np.nan, "Missing haemoglobin")
seed_error(37, "ICDT", pd.NaT, "Missing consent date")
seed_error(48, "VISDT", pd.NaT, "Missing visit date")
seed_error(62, "SBP", np.nan, "Missing systolic BP")
seed_error(74, "WT", np.nan, "Missing weight")

# Out-of-range values
seed_error(13, "SBP", 225, "SBP above assumed maximum")
seed_error(21, "DBP", 130, "DBP above assumed maximum")
seed_error(31, "HT", 125.0, "Height below assumed minimum")
seed_error(43, "WT", 720.0, "Implausible weight; confirm value/unit")
seed_error(56, "HGB", 22.5, "HGB above assumed maximum")
seed_error(68, "AGE", 82, "Age above assumed maximum")

# Visit before consent
seed_error(
    8, "VISDT",
    df.loc[8, "ICDT"] - pd.Timedelta(days=2),
    "Visit date before consent"
)

# Duplicate subject ID
seed_error(
    90, "SUBJID",
    df.loc[89, "SUBJID"],
    "Duplicate subject ID"
)

# Inconsistent sex coding
seed_error(15, "SEX", "Male", "Non-standard sex coding")
seed_error(26, "SEX", "male", "Non-standard sex coding")
seed_error(39, "SEX", "Female", "Non-standard sex coding")

# Unit inconsistency: weight expressed in pounds but stored in WT
pounds = round(float(df.loc[52, "WT"]) * 2.20462, 1)
seed_error(
    52, "WT", pounds,
    "Weight may be in pounds instead of kg; confirm before conversion"
)

# Save the raw synthetic dataset
raw_path = DATA_DIR / "synthetic_raw.csv"
df.to_csv(raw_path, index=False, date_format="%Y-%m-%d")

# Save a development manifest of intentionally seeded issues
manifest_path = DATA_DIR / "seeded_issue_manifest.csv"
pd.DataFrame(seeded_issues).to_csv(manifest_path, index=False)

# Display results
print("Dataset generation completed.")
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))
print("Intentionally seeded issues:", len(seeded_issues))
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

print("\nFiles saved:")
print(raw_path)
print(manifest_path)


Dataset generation completed.
Number of rows: 100
Number of columns: 12
Intentionally seeded issues: 19

Columns:
['SUBJID', 'SITE', 'ICDT', 'VISDT', 'AGE', 'SEX', 'HT', 'WT', 'SBP', 'DBP', 'HGB', 'AETERM']

First five rows:


,SUBJID,SITE,ICDT,VISDT,AGE,SEX,HT,WT,SBP,DBP,HGB,AETERM
0,S001,101,2025-10-07,2025-10-18,22.0,M,155.0,90.6,136.0,98,13.1,
1,S002,103,2025-03-20,2025-03-27,38.0,F,189.1,112.2,112.0,72,13.0,
2,S003,102,2025-10-12,2025-10-15,NaN,F,182.2,83.3,153.0,68,16.0,
3,S004,102,2025-01-10,2025-01-13,60.0,M,151.2,45.3,133.0,97,11.7,
4,S005,103,2025-05-21,2025-05-26,36.0,M,192.3,81.8,155.0,64,10.9,



Files saved:
/content/Clinical-Data-Cleaning-Project/data/synthetic_raw.csv
/content/Clinical-Data-Cleaning-Project/data/seeded_issue_manifest.csv


In [4]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path("/content/Clinical-Data-Cleaning-Project")
DATA_DIR = PROJECT_DIR / "data"

raw_path = DATA_DIR / "synthetic_raw.csv"
manifest_path = DATA_DIR / "seeded_issue_manifest.csv"

# Load the saved files
df = pd.read_csv(raw_path, parse_dates=["ICDT", "VISDT"])
seeded_issues_df = pd.read_csv(manifest_path)

print("Dataset loaded successfully.")
print("Total rows:", len(df))
print("Total columns:", len(df.columns))
print("Seeded issues recorded:", len(seeded_issues_df))

Dataset loaded successfully.
Total rows: 100
Total columns: 12
Seeded issues recorded: 19


In [5]:
print("DATASET VERIFICATION")
print("-" * 40)

print("Total rows:", len(df))
print("Total columns:", len(df.columns))
print("Seeded issues recorded:", len(seeded_issues_df))

print("\nMissing values per column:")
print(df.isna().sum())

print("\nDuplicate Subject IDs:")
print(df.loc[df["SUBJID"].duplicated(keep=False), "SUBJID"].tolist())

print("\nDate chronology violations:")
date_issue = (
    df["ICDT"].notna()
    & df["VISDT"].notna()
    & (df["VISDT"] < df["ICDT"])
)
print("Count:", int(date_issue.sum()))

print("\nSex values present:")
print(df["SEX"].value_counts(dropna=False))

print("\nWeights above the assumed maximum of 150 kg:")
print(df.loc[df["WT"] > 150, ["SUBJID", "WT"]])

assert len(df) == 100, "Expected 100 rows"
assert len(df.columns) == 12, "Expected 12 columns"
assert len(seeded_issues_df) == 19, "Expected 19 seeded issues"

print("\nVerification completed successfully.")

DATASET VERIFICATION
----------------------------------------
Total rows: 100
Total columns: 12
Seeded issues recorded: 19

Missing values per column:
SUBJID     0
SITE       0
ICDT       1
VISDT      1
AGE        1
SEX        0
HT         1
WT         1
SBP        1
DBP        0
HGB        1
AETERM    51
dtype: int64

Duplicate Subject IDs:
['S090', 'S090']

Date chronology violations:
Count: 1

Sex values present:
SEX
M         49
F         48
Male       1
male       1
Female     1
Name: count, dtype: int64

Weights above the assumed maximum of 150 kg:
   SUBJID     WT
43   S044  720.0
52   S053  223.5

Verification completed successfully.


In [6]:
from google.colab import files

files.download(str(raw_path))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
files.download(str(manifest_path))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:

from pathlib import Path
import shutil

source = Path("/content/Data_Dictionary.xlsx")
destination = Path(
    "/content/Clinical-Data-Cleaning-Project/Data_Dictionary.xlsx"
)

if source.exists():
    shutil.copy2(source, destination)
    print("Data_Dictionary.xlsx copied successfully!")
else:
    print("File not found. Please upload Data_Dictionary.xlsx first.")


File not found. Please upload Data_Dictionary.xlsx first.


In [15]:

from pathlib import Path

project = Path("/content/Clinical-Data-Cleaning-Project")

required_files = [
    project / "data" / "synthetic_raw.csv",
    project / "data" / "seeded_issue_manifest.csv",
    project / "Data_Dictionary.xlsx"
]

for file in required_files:
    print(("FOUND" if file.exists() else "MISSING"), "-", file.name)


FOUND - synthetic_raw.csv
FOUND - seeded_issue_manifest.csv
MISSING - Data_Dictionary.xlsx


In [16]:

import pandas as pd
from pathlib import Path

# Define project folders
PROJECT_DIR = Path("/content/Clinical-Data-Cleaning-Project")
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load raw synthetic data
df = pd.read_csv(
    DATA_DIR / "synthetic_raw.csv",
    parse_dates=["ICDT", "VISDT"]
)

# Preserve the original data
raw_df = df.copy(deep=True)

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nColumn names:")
print(df.columns.tolist())

display(df.head())


Dataset loaded successfully!
Rows: 100
Columns: 12

Column names:
['SUBJID', 'SITE', 'ICDT', 'VISDT', 'AGE', 'SEX', 'HT', 'WT', 'SBP', 'DBP', 'HGB', 'AETERM']


,SUBJID,SITE,ICDT,VISDT,AGE,SEX,HT,WT,SBP,DBP,HGB,AETERM
0,S001,101,2025-10-07,2025-10-18,22.0,M,155.0,90.6,136.0,98,13.1,NaN
1,S002,103,2025-03-20,2025-03-27,38.0,F,189.1,112.2,112.0,72,13.0,NaN
2,S003,102,2025-10-12,2025-10-15,NaN,F,182.2,83.3,153.0,68,16.0,NaN
3,S004,102,2025-01-10,2025-01-13,60.0,M,151.2,45.3,133.0,97,11.7,NaN
4,S005,103,2025-05-21,2025-05-26,36.0,M,192.3,81.8,155.0,64,10.9,NaN


In [17]:

# Store every detected discrepancy here
issues = []

def add_issue(row_index, check_id, variable, observed_value, description, query):
    issues.append({
        "SUBJID": raw_df.loc[row_index, "SUBJID"],
        "CHECK_ID": check_id,
        "VARIABLE": variable,
        "OBSERVED_VALUE": str(observed_value),
        "ISSUE_DESCRIPTION": description,
        "DRAFT_SITE_QUERY": query
    })

# EC01: Missing Subject ID
for i in raw_df.index[raw_df["SUBJID"].isna() |
                      raw_df["SUBJID"].astype(str).str.strip().eq("")]:
    add_issue(i, "EC01", "SUBJID", raw_df.loc[i, "SUBJID"],
              "Subject ID is missing",
              "Please confirm and provide the correct subject identifier.")

# EC02: Duplicate Subject ID
duplicate_mask = raw_df["SUBJID"].notna() & raw_df["SUBJID"].duplicated(keep=False)
for i in raw_df.index[duplicate_mask]:
    add_issue(i, "EC02", "SUBJID", raw_df.loc[i, "SUBJID"],
              "Subject ID is duplicated",
              "Please verify the subject identifier and confirm whether this record is a duplicate.")

# EC03: Missing consent date
for i in raw_df.index[raw_df["ICDT"].isna()]:
    add_issue(i, "EC03", "ICDT", raw_df.loc[i, "ICDT"],
              "Consent date is missing",
              "Please verify the informed consent date against source documentation.")

# EC04: Missing visit date
for i in raw_df.index[raw_df["VISDT"].isna()]:
    add_issue(i, "EC04", "VISDT", raw_df.loc[i, "VISDT"],
              "Visit date is missing",
              "Please verify the visit date against source documentation.")

# EC05: Visit date before consent date
mask = raw_df["ICDT"].notna() & raw_df["VISDT"].notna() & (
    raw_df["VISDT"] < raw_df["ICDT"]
)
for i in raw_df.index[mask]:
    add_issue(i, "EC05", "VISDT", raw_df.loc[i, "VISDT"],
              "Visit date precedes consent date",
              "Please confirm both dates and clarify the chronology.")

# EC06: Missing or out-of-range age
age = pd.to_numeric(raw_df["AGE"], errors="coerce")
mask = age.isna() | ~age.between(18, 75)
for i in raw_df.index[mask]:
    add_issue(i, "EC06", "AGE", raw_df.loc[i, "AGE"],
              "Age is missing, invalid, or outside 18–75 years",
              "Please verify the recorded age and confirm the correct value.")

# EC07: Non-standard sex coding
sex = raw_df["SEX"].astype("string").str.strip()
mask = sex.isna() | ~sex.isin(["M", "F"])
for i in raw_df.index[mask]:
    add_issue(i, "EC07", "SEX", raw_df.loc[i, "SEX"],
              "Sex coding is missing or non-standard",
              "Please confirm the intended coding using the study data conventions.")

# EC08: Missing or out-of-range height
height = pd.to_numeric(raw_df["HT"], errors="coerce")
mask = height.isna() | ~height.between(140, 200)
for i in raw_df.index[mask]:
    add_issue(i, "EC08", "HT", raw_df.loc[i, "HT"],
              "Height is missing, invalid, or outside 140–200 cm",
              "Please verify the height and confirm the unit is centimetres.")

# EC09: Missing or out-of-range weight
weight = pd.to_numeric(raw_df["WT"], errors="coerce")
mask = weight.isna() | ~weight.between(40, 150)
for i in raw_df.index[mask]:
    add_issue(i, "EC09", "WT", raw_df.loc[i, "WT"],
              "Weight is missing, invalid, or outside 40–150 kg",
              "Please verify the weight and confirm the unit is kilograms before any conversion.")

# EC10: Missing or out-of-range systolic blood pressure
sbp = pd.to_numeric(raw_df["SBP"], errors="coerce")
mask = sbp.isna() | ~sbp.between(80, 200)
for i in raw_df.index[mask]:
    add_issue(i, "EC10", "SBP", raw_df.loc[i, "SBP"],
              "Systolic blood pressure is missing, invalid, or outside 80–200 mmHg",
              "Please verify the systolic blood pressure against the source record.")

# EC11: Missing or invalid diastolic blood pressure
dbp = pd.to_numeric(raw_df["DBP"], errors="coerce")
mask = (
    dbp.isna()
    | ~dbp.between(50, 120)
    | (sbp.notna() & dbp.notna() & (dbp >= sbp))
)
for i in raw_df.index[mask]:
    add_issue(i, "EC11", "DBP", raw_df.loc[i, "DBP"],
              "Diastolic blood pressure is missing, outside 50–120 mmHg, or not below SBP",
              "Please verify both blood pressure values and confirm the correct measurements.")

# EC12: Missing or out-of-range haemoglobin
hgb = pd.to_numeric(raw_df["HGB"], errors="coerce")
mask = hgb.isna() | ~hgb.between(8, 18)
for i in raw_df.index[mask]:
    add_issue(i, "EC12", "HGB", raw_df.loc[i, "HGB"],
              "Haemoglobin is missing, invalid, or outside 8–18 g/dL",
              "Please verify the haemoglobin result and confirm the unit is g/dL.")

# Convert discrepancies into a table
discrepancy_df = pd.DataFrame(
    issues,
    columns=[
        "SUBJID", "CHECK_ID", "VARIABLE",
        "OBSERVED_VALUE", "ISSUE_DESCRIPTION", "DRAFT_SITE_QUERY"
    ]
)

print("Edit checks completed.")
print("Total discrepancy records:", len(discrepancy_df))
display(discrepancy_df.head(10))


Edit checks completed.
Total discrepancy records: 20


,SUBJID,CHECK_ID,VARIABLE,OBSERVED_VALUE,ISSUE_DESCRIPTION,DRAFT_SITE_QUERY
0,S090,EC02,SUBJID,S090,Subject ID is duplicated,Please verify the subject identifier and confi...
1,S090,EC02,SUBJID,S090,Subject ID is duplicated,Please verify the subject identifier and confi...
2,S038,EC03,ICDT,NaT,Consent date is missing,Please verify the informed consent date agains...
3,S049,EC04,VISDT,NaT,Visit date is missing,Please verify the visit date against source do...
4,S009,EC05,VISDT,2025-08-26 00:00:00,Visit date precedes consent date,Please confirm both dates and clarify the chro...
5,S003,EC06,AGE,nan,"Age is missing, invalid, or outside 18–75 years",Please verify the recorded age and confirm the...
6,S069,EC06,AGE,82.0,"Age is missing, invalid, or outside 18–75 years",Please verify the recorded age and confirm the...
7,S016,EC07,SEX,Male,Sex coding is missing or non-standard,Please confirm the intended coding using the s...
8,S027,EC07,SEX,male,Sex coding is missing or non-standard,Please confirm the intended coding using the s...
9,S040,EC07,SEX,Female,Sex coding is missing or non-standard,Please confirm the intended coding using the s...


In [18]:

# Save discrepancy log
discrepancy_path = OUTPUT_DIR / "discrepancy_log.csv"
discrepancy_df.to_csv(discrepancy_path, index=False)

# Save a summary of findings by edit check
summary_df = (
    discrepancy_df.groupby("CHECK_ID")
    .size()
    .reset_index(name="DISCREPANCY_COUNT")
    .sort_values("CHECK_ID")
)

summary_path = OUTPUT_DIR / "edit_check_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("Files saved successfully:")
print(discrepancy_path)
print(summary_path)

print("\nDiscrepancies by edit check:")
display(summary_df)

print("\nTotal discrepancy records:", len(discrepancy_df))
print("Unique subjects affected:", discrepancy_df["SUBJID"].nunique())


Files saved successfully:
/content/Clinical-Data-Cleaning-Project/outputs/discrepancy_log.csv
/content/Clinical-Data-Cleaning-Project/outputs/edit_check_summary.csv

Discrepancies by edit check:


,CHECK_ID,DISCREPANCY_COUNT
0,EC02,2
1,EC03,1
2,EC04,1
3,EC05,1
4,EC06,2
5,EC07,3
6,EC08,2
7,EC09,3
8,EC10,2
9,EC11,1



Total discrepancy records: 20
Unique subjects affected: 19


In [19]:

print("EDIT CHECK SUMMARY")
print("=" * 40)

display(
    summary_df.sort_values(
        "DISCREPANCY_COUNT",
        ascending=False
    )
)

print("\nTotal discrepancy records:", len(discrepancy_df))
print("Unique subjects affected:", discrepancy_df["SUBJID"].nunique())
print("Edit checks with findings:", discrepancy_df["CHECK_ID"].nunique())


EDIT CHECK SUMMARY


,CHECK_ID,DISCREPANCY_COUNT
5,EC07,3
7,EC09,3
0,EC02,2
10,EC12,2
8,EC10,2
6,EC08,2
4,EC06,2
2,EC04,1
1,EC03,1
3,EC05,1



Total discrepancy records: 20
Unique subjects affected: 19
Edit checks with findings: 11


In [20]:

print("SAVED OUTPUT FILES")
print("=" * 40)

for file in sorted(OUTPUT_DIR.glob("*")):
    if file.is_file():
        print(f"{file.name} — {file.stat().st_size:,} bytes")


SAVED OUTPUT FILES
discrepancy_log.csv — 2,832 bytes
edit_check_summary.csv — 104 bytes


In [21]:

display(
    discrepancy_df[
        [
            "SUBJID",
            "CHECK_ID",
            "VARIABLE",
            "OBSERVED_VALUE",
            "ISSUE_DESCRIPTION",
            "DRAFT_SITE_QUERY"
        ]
    ].head(10)
)


,SUBJID,CHECK_ID,VARIABLE,OBSERVED_VALUE,ISSUE_DESCRIPTION,DRAFT_SITE_QUERY
0,S090,EC02,SUBJID,S090,Subject ID is duplicated,Please verify the subject identifier and confi...
1,S090,EC02,SUBJID,S090,Subject ID is duplicated,Please verify the subject identifier and confi...
2,S038,EC03,ICDT,NaT,Consent date is missing,Please verify the informed consent date agains...
3,S049,EC04,VISDT,NaT,Visit date is missing,Please verify the visit date against source do...
4,S009,EC05,VISDT,2025-08-26 00:00:00,Visit date precedes consent date,Please confirm both dates and clarify the chro...
5,S003,EC06,AGE,nan,"Age is missing, invalid, or outside 18–75 years",Please verify the recorded age and confirm the...
6,S069,EC06,AGE,82.0,"Age is missing, invalid, or outside 18–75 years",Please verify the recorded age and confirm the...
7,S016,EC07,SEX,Male,Sex coding is missing or non-standard,Please confirm the intended coding using the s...
8,S027,EC07,SEX,male,Sex coding is missing or non-standard,Please confirm the intended coding using the s...
9,S040,EC07,SEX,Female,Sex coding is missing or non-standard,Please confirm the intended coding using the s...


In [22]:

from google.colab import files

files.download(
    "/content/Clinical-Data-Cleaning-Project/outputs/discrepancy_log.csv"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:

from google.colab import files

files.download(
    "/content/Clinical-Data-Cleaning-Project/outputs/edit_check_summary.csv"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:

from pathlib import Path
from datetime import date
import pandas as pd

PROJECT_DIR = Path("/content/Clinical-Data-Cleaning-Project")
REPORTS_DIR = PROJECT_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Build a summary using the actual edit-check results
total_rows = len(df)
total_columns = len(df.columns)
total_discrepancies = len(discrepancy_df)
unique_subjects_affected = discrepancy_df["SUBJID"].nunique()
checks_with_findings = discrepancy_df["CHECK_ID"].nunique()

report_lines = [
    "# Clinical Data Quality Report",
    "",
    "## 1. Project Overview",
    "",
    "This report summarizes data quality checks performed on a synthetic clinical study dataset using Python and pandas.",
    "",
    "## 2. Dataset Summary",
    "",
    f"- Total records: {total_rows}",
    f"- Total variables: {total_columns}",
    "- Data source: Synthetic data created for educational purposes",
    "",
    "## 3. Validation Approach",
    "",
    "Twelve edit checks were implemented to identify missing identifiers and dates, duplicate subject IDs, date chronology issues, invalid demographic coding, and values outside the project's assumed validation ranges.",
    "",
    "## 4. Results",
    "",
    f"- Total discrepancy records: {total_discrepancies}",
    f"- Unique subjects affected: {unique_subjects_affected}",
    f"- Edit checks with findings: {checks_with_findings}",
    "",
    "### Findings by Edit Check",
    "",
    "| Check ID | Discrepancy Records |",
    "|---|---:|",
]

for _, row in summary_df.sort_values("CHECK_ID").iterrows():
    report_lines.append(
        f"| {row['CHECK_ID']} | {int(row['DISCREPANCY_COUNT'])} |"
    )

report_lines += [
    "",
    "## 5. Discrepancy Management",
    "",
    "Findings were recorded in a discrepancy log with the subject identifier, check ID, variable, observed value, issue description, and draft site query. The log is intended to support review and clarification. Values were not automatically corrected.",
    "",
    "## 6. Limitations",
    "",
    "- All records are synthetic; this analysis does not represent a real clinical trial.",
    "- Validation ranges are assumed rules for this portfolio exercise, not universal clinical reference ranges.",
    "- An out-of-range value is a signal for investigation, not proof that the value is incorrect.",
    "- Potential unit inconsistencies require confirmation before conversion.",
    "- A discrepancy record is not necessarily a unique affected subject.",
    "",
    "## 7. Conclusion",
    "",
    f"The workflow processed {total_rows} synthetic records and generated {total_discrepancies} discrepancy records affecting {unique_subjects_affected} unique subjects. The project demonstrates a basic, reproducible approach to clinical data validation and discrepancy documentation.",
    "",
    "## 8. Tools",
    "",
    "Python, pandas, Google Colab, Excel, and GitHub.",
    "",
    "## 9. Data Privacy",
    "",
    "No real patient data were used. All data were generated for educational and portfolio purposes.",
    "",
    f"Report generated: {date.today().isoformat()}",
]

report_path = REPORTS_DIR / "Data_Quality_Report.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")

print("Report created successfully:")
print(report_path)
print("\nPreview:")
print(report_path.read_text(encoding="utf-8")[:2500])


Report created successfully:
/content/Clinical-Data-Cleaning-Project/reports/Data_Quality_Report.md

Preview:
# Clinical Data Quality Report

## 1. Project Overview

This report summarizes data quality checks performed on a synthetic clinical study dataset using Python and pandas.

## 2. Dataset Summary

- Total records: 100
- Total variables: 12
- Data source: Synthetic data created for educational purposes

## 3. Validation Approach

Twelve edit checks were implemented to identify missing identifiers and dates, duplicate subject IDs, date chronology issues, invalid demographic coding, and values outside the project's assumed validation ranges.

## 4. Results

- Total discrepancy records: 20
- Unique subjects affected: 19
- Edit checks with findings: 11

### Findings by Edit Check

| Check ID | Discrepancy Records |
|---|---:|
| EC02 | 2 |
| EC03 | 1 |
| EC04 | 1 |
| EC05 | 1 |
| EC06 | 2 |
| EC07 | 3 |
| EC08 | 2 |
| EC09 | 3 |
| EC10 | 2 |
| EC11 | 1 |
| EC12 | 2 |

## 5. Discrepanc

In [25]:

from google.colab import files

files.download(
    "/content/Clinical-Data-Cleaning-Project/reports/Data_Quality_Report.md"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>